# Obstacle recognition v3: synthetic warehouse + real LOCO photos (YOLO11n, Kaggle)

Classes: **person, pallet, cart, forklift** (pallet jacks and pallet trucks count as forklifts).

Training data:
- `obstacles_v3`: 12,000 Gazebo images (textured pallets, built forklifts and pallet jacks, pallet racking, colour-varied worlds, camera effects, no sliver labels).
- `loco_yolo` (train/val part): real photos from three logistics sites, LOCO (TU Munich, CC0); people labelled by the COCO YOLO11m.

Held-out tests (never trained on):
- `obstacles_v3_test`: unseen people, pallet, carts **and forklift / pallet jack** models.
- `obstacles_v1_test`: the test v1 and v2 were scored on (for comparison).
- `loco_yolo` test part: **real photos from two other sites** (LOCO subsets 1 and 4): pallet and forklift only.

**Settings** (right panel): Accelerator **GPU T4 x2** (one is used) or P100, Internet **On**; **Add Input**: your dataset `tugger-obstacles-v3` with `obstacles_v3.zip`, `obstacles_v3_test.zip`, `loco_yolo.zip`, `obstacles_v1_test.zip`.

Run the cells in order. Training takes about 4-5 hours; the run is saved in `/kaggle/working/runs`. If the session stops: run cells 1 and 2, then cell 5 (resume).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics==8.3.40

In [ ]:
# Find the uploads, copy them (32 files at a time: Kaggle input is slow per file).
import glob, os, shutil, zipfile, yaml, time
from concurrent.futures import ThreadPoolExecutor
IN, TMP, DATA = '/kaggle/input', '/kaggle/temp_unzip', '/content'
for z in glob.glob(f'{IN}/**/*.zip', recursive=True):
    zipfile.ZipFile(z).extractall(TMP)

def find(name, inner):
    for root in (IN, TMP):
        for d, _, _ in os.walk(root):
            if os.path.basename(d) == name and os.path.isdir(f'{d}/{inner}'):
                return d
    raise FileNotFoundError(name)

def copy_one(pair):
    s, d = pair
    if not (os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s)):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copyfile(s, d)

for name, inner in (('obstacles_v3', 'images/train'), ('obstacles_v3_test', 'images/test'),
                    ('obstacles_v1_test', 'images/test'), ('loco_yolo', 'images/train')):
    src, dst = find(name, inner), f'{DATA}/{name}'
    pairs = [(os.path.join(d, f), os.path.join(dst, os.path.relpath(os.path.join(d, f), src)))
             for d, _, files in os.walk(src) for f in files]
    t = time.time()
    with ThreadPoolExecutor(32) as pool:
        list(pool.map(copy_one, pairs))
    print(f'{name}: {len(pairs)} files in {time.time() - t:.0f} s')

names = {0: 'person', 1: 'pallet', 2: 'cart', 3: 'forklift'}
yaml.safe_dump({'path': DATA, 'names': names,
                'train': ['obstacles_v3/images/train', 'loco_yolo/images/train'],
                'val': ['obstacles_v3/images/val', 'loco_yolo/images/val']}, open(f'{DATA}/train_v3.yaml', 'w'))
for test, folder in (('obstacles_v3_test', 'obstacles_v3_test/images/test'),
                     ('obstacles_v1_test', 'obstacles_v1_test/images/test'),
                     ('loco_test', 'loco_yolo/images/test')):
    yaml.safe_dump({'path': DATA, 'names': names, 'train': folder, 'val': folder},
                   open(f'{DATA}/{test}.yaml', 'w'))
for d in ('obstacles_v3/images/train', 'obstacles_v3/images/val', 'loco_yolo/images/train',
          'loco_yolo/images/val', 'loco_yolo/images/test', 'obstacles_v3_test/images/test',
          'obstacles_v1_test/images/test'):
    print(d, len(os.listdir(f'{DATA}/{d}')))

In [ ]:
# Shared set-up for the training cells (Kaggle's Ray version breaks Ultralytics' Ray Tune link).
import numpy as np
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid
from ultralytics import YOLO
import ultralytics.utils.callbacks.raytune as raytune
raytune.callbacks.clear()
RUNS = '/kaggle/working/runs'

In [ ]:
# Train from the COCO weights. Same settings as v1/v2 (only the data changed), early stop after 20 epochs without gain.
model = YOLO('yolo11n.pt')
model.train(data='/content/train_v3.yaml', imgsz=640, epochs=100, batch=32,
            patience=20, fliplr=0.5, flipud=0.0, degrees=0.0, mosaic=1.0, scale=0.5,
            translate=0.1, hsv_h=0.02, hsv_s=0.6, hsv_v=0.5, weight_decay=0.0005,
            project=RUNS, name='obstacles_v3', seed=11, deterministic=True)

In [ ]:
# ONLY if the session stopped: run cells 1, 2, 3 and this one; it continues from the last epoch.
YOLO(f'{RUNS}/obstacles_v3/weights/last.pt').train(resume=True)

In [ ]:
# Overfitting check: training and validation losses side by side.
import pandas as pd, matplotlib.pyplot as plt
r = pd.read_csv(f'{RUNS}/obstacles_v3/results.csv')
r.columns = [c.strip() for c in r.columns]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, name in zip(axes, ['box_loss', 'cls_loss']):
    ax.plot(r['epoch'], r[f'train/{name}'], label='train')
    ax.plot(r['epoch'], r[f'val/{name}'], label='validation')
    ax.set_title(name); ax.set_xlabel('epoch'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RUNS}/obstacles_v3/overfitting_check.png', dpi=150); plt.show()

In [ ]:
# Scores: validation, and the three held-out tests (per class).
import json
best = YOLO(f'{RUNS}/obstacles_v3/weights/best.pt')
summary = {}
for name, data in (('val', '/content/train_v3.yaml'), ('v3_test', '/content/obstacles_v3_test.yaml'),
                   ('v1_test', '/content/obstacles_v1_test.yaml'), ('loco_real_test', '/content/loco_test.yaml')):
    r = best.val(data=data, imgsz=640, project=RUNS, name=f'obstacles_v3_{name}')
    summary[name] = {'mAP50-95': float(r.box.map), 'mAP50': float(r.box.map50),
                     'per_class_mAP50-95': {r.names[c]: float(m) for c, m in zip(r.box.ap_class_index, r.box.maps[r.box.ap_class_index])}}
print(json.dumps(summary, indent=1))
json.dump(summary, open(f'{RUNS}/obstacles_v3/summary.json', 'w'), indent=1)

In [ ]:
!cd {RUNS} && zip -q -r /kaggle/working/obstacle_detector_v3.zip . -x '*/weights/last.pt'
from IPython.display import FileLink
FileLink('obstacle_detector_v3.zip')   # click to download (also under Output on the right)